# Notebook 27 — Entrenamiento Modelo D (YOLOv8m-seg)

**TFM — Sistema de Detección de Amenazas Armadas en Vídeo**  
Oliver Legarreta García · Universitat Oberta de Catalunya

---

## Objetivo

Entrenar **Modelo D** — variante mejorada de Modelo C con negativos difíciles equilibrados — y compararlo con Modelo B y Modelo C sobre el dataset GAR.

## Diferencia respecto a Modelo C

| Aspecto | Modelo C | Modelo D |
|---------|---------|----------|
| Arquitectura | yolov8m-seg | yolov8m-seg |
| Positivos | 8.246 armas (Roboflow) | 8.246 armas (Roboflow) |
| Negativos | 108 (LVIS) | **1.586** (LVIS + Open Images V7) |
| Ratio pos/neg | 76:1 | **5.2:1** |
| Clases negativas | teléfono, botella, mando | + **persona, mobile phone** |

## Hipótesis

El desequilibrio extremo de Modelo C (76:1) hacía que el modelo apenas viera negativos durante el entrenamiento, generando FP elevados en teléfonos (+50pp en N5) y personas sin objeto (+28pp en N4). Con un ratio 5:1 y negativos más representativos, el modelo debería mejorar la discriminación en esas categorías.

---
## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install ultralytics
import ultralytics
ultralytics.checks()
print('✅ Ultralytics instalado')

In [ ]:
import os
import shutil
import yaml
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

# ── CONFIG ────────────────────────────────────────────────────────────────────
GUN_DS   = '/content/Gun-Project-1'          # Roboflow — descargar si no existe
LVIS_DS  = '/content/drive/MyDrive/TFM/datasets/lvis_negatives'
OI_DS    = '/content/drive/MyDrive/TFM/datasets/openimages_negatives'
COMBINED = '/content/dataset_modelo_d'
OUT_DIR  = '/content/drive/MyDrive/TFM/experiments/weapon_seg/yolov8m_seg_D'

EPOCHS   = 50
IMG_SIZE = 640
BATCH    = 16
PATIENCE = 15

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print('✅ Config cargada')
print(f'   Epochs:   {EPOCHS}')
print(f'   Batch:    {BATCH}')
print(f'   ImgSize:  {IMG_SIZE}')

---
## 1. Descargar dataset Roboflow (si no existe en /content/)

In [ ]:
if not Path(GUN_DS).exists():
    print('Descargando dataset Roboflow...')
    !pip -q install roboflow
    from roboflow import Roboflow
    rf      = Roboflow(api_key="TU_API_KEY")  # ← sustituir por tu API key
    project = rf.workspace("gun-iwmih").project("gun-project-nzlce")
    version = project.version(1)
    dataset = version.download("yolov8")
    print('✅ Dataset Roboflow descargado')
else:
    print('✅ Dataset Roboflow ya existe en /content/')

# Verificar
for split in ['train', 'valid', 'test']:
    imgs = list((Path(GUN_DS) / split / 'images').glob('*'))
    print(f'  {split}: {len(imgs)} imágenes')

---
## 2. Construir dataset combinado

In [ ]:
if Path(COMBINED).exists():
    shutil.rmtree(COMBINED)
    print('  Carpeta anterior eliminada')

for split in ['train', 'valid', 'test']:
    (Path(COMBINED) / split / 'images').mkdir(parents=True)
    (Path(COMBINED) / split / 'labels').mkdir(parents=True)

# ── Copiar positivos Roboflow ──────────────────────────────────────────────────
print('Copiando positivos Roboflow...')
for split, src_split in [('train','train'), ('valid','valid'), ('test','test')]:
    src_img = Path(GUN_DS) / src_split / 'images'
    src_lbl = Path(GUN_DS) / src_split / 'labels'
    dst_img = Path(COMBINED) / split / 'images'
    dst_lbl = Path(COMBINED) / split / 'labels'
    n = 0
    for img in src_img.glob('*'):
        shutil.copy2(img, dst_img / img.name)
        lbl = src_lbl / (img.stem + '.txt')
        if lbl.exists():
            shutil.copy2(lbl, dst_lbl / lbl.name)
        n += 1
    print(f'  {split}: {n} imágenes')

# ── Copiar negativos LVIS ─────────────────────────────────────────────────────
print('\nAñadiendo negativos LVIS...')
lvis_imgs = list((Path(LVIS_DS) / 'images' / 'train').glob('*.jpg'))
dst_img   = Path(COMBINED) / 'train' / 'images'
dst_lbl   = Path(COMBINED) / 'train' / 'labels'
for img in lvis_imgs:
    shutil.copy2(img, dst_img / img.name)
    lbl = Path(LVIS_DS) / 'labels' / 'train' / (img.stem + '.txt')
    if lbl.exists():
        shutil.copy2(lbl, dst_lbl / lbl.name)
    else:
        (dst_lbl / (img.stem + '.txt')).write_text('')
print(f'  train: {len(lvis_imgs)} negativos LVIS añadidos')

# ── Copiar negativos Open Images V7 ──────────────────────────────────────────
print('\nAñadiendo negativos Open Images V7...')
oi_imgs = list((Path(OI_DS) / 'images' / 'train').glob('*'))
for img in oi_imgs:
    shutil.copy2(img, dst_img / img.name)
    lbl = Path(OI_DS) / 'labels' / 'train' / (img.stem + '.txt')
    if lbl.exists():
        shutil.copy2(lbl, dst_lbl / lbl.name)
    else:
        (dst_lbl / (img.stem + '.txt')).write_text('')
print(f'  train: {len(oi_imgs)} negativos OI V7 añadidos')

# ── Verificar totales ─────────────────────────────────────────────────────────
print()
print('=== Dataset combinado final ===')
for split in ['train', 'valid', 'test']:
    imgs = list((Path(COMBINED) / split / 'images').glob('*'))
    lbls = list((Path(COMBINED) / split / 'labels').glob('*.txt'))
    print(f'  {split:<6}: {len(imgs):>5} imágenes | {len(lbls):>5} labels')

train_imgs = list((Path(COMBINED) / 'train' / 'images').glob('*'))
n_pos = len(list((Path(GUN_DS) / 'train' / 'images').glob('*')))
n_neg = len(lvis_imgs) + len(oi_imgs)
print(f'\n  Positivos train: {n_pos}')
print(f'  Negativos train: {n_neg} (LVIS: {len(lvis_imgs)} + OI V7: {len(oi_imgs)})')
print(f'  Ratio pos/neg:   {n_pos/n_neg:.1f}')

In [ ]:
# Limpiar labels con mezcla bbox/segmentación y generar data.yaml
print('Limpiando labels con polígonos inválidos...')
lbl_dir = Path(COMBINED) / 'train' / 'labels'
removed, fixed = 0, 0
for lbl_path in lbl_dir.glob('*.txt'):
    if lbl_path.stat().st_size == 0:
        continue
    lines = lbl_path.read_text().strip().split('\n')
    valid = [l for l in lines if len(l.strip().split()) >= 7]
    if len(valid) == 0:
        lbl_path.write_text(''); removed += 1
    elif len(valid) < len(lines):
        lbl_path.write_text('\n'.join(valid)); fixed += 1
for cache in Path(COMBINED).rglob('*.cache'):
    cache.unlink()
print(f'  Labels vaciados: {removed} | Labels corregidos: {fixed} | Caches eliminados')

# data.yaml
data_yaml = {
    'path': COMBINED,
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    'nc': 1,
    'names': {0: 'gun'},
}
yaml_path = Path(COMBINED) / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, allow_unicode=True)
print(f'✅ data.yaml generado')
print(open(yaml_path).read())

---
## 3. Verificación visual del dataset

In [ ]:
train_img_dir = Path(COMBINED) / 'train' / 'images'
train_lbl_dir = Path(COMBINED) / 'train' / 'labels'

# 4 positivos + 4 LVIS + 4 OI V7
gun_imgs  = [f for f in sorted(train_img_dir.glob('*'))
             if not f.name.startswith('lvis_') and not f.name.startswith('oi_')][:4]
lvis_imgs_sample = [f for f in sorted(train_img_dir.glob('lvis_*.jpg'))][:4]
oi_imgs_sample   = [f for f in sorted(train_img_dir.glob('oi_*.jpg'))][:4]

fig, axes = plt.subplots(3, 4, figsize=(16, 9))
row_labels = ['Positivos (armas)', 'Negativos LVIS', 'Negativos OI V7']
COLORS = [(255,80,80),(80,255,80),(80,80,255),(255,255,80)]

for row, img_list in enumerate([gun_imgs, lvis_imgs_sample, oi_imgs_sample]):
    for col, img_path in enumerate(img_list):
        ax  = axes[row, col]
        lbl = train_lbl_dir / (img_path.stem + '.txt')
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        overlay = img.copy()
        if lbl.exists() and lbl.stat().st_size > 0:
            for li, line in enumerate(lbl.read_text().strip().split('\n')):
                parts = line.strip().split()
                if len(parts) < 7: continue
                coords = list(map(float, parts[1:]))
                pts = np.array([[coords[i]*w, coords[i+1]*h]
                                for i in range(0, len(coords), 2)], dtype=np.int32)
                cv2.fillPoly(overlay, [pts], COLORS[li % len(COLORS)])
            img = cv2.addWeighted(img, 0.55, overlay, 0.45, 0)
        ax.imshow(img)
        if col == 0:
            ax.set_ylabel(row_labels[row], fontsize=9)
        ax.axis('off')

plt.suptitle('Muestra dataset Modelo D — positivos y negativos con máscaras', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/muestra_dataset_D.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Muestra guardada')

---
## 4. Entrenamiento Modelo D

In [ ]:
model = YOLO('yolov8m-seg.pt')

print('Iniciando entrenamiento Modelo D...')
print(f'  Arquitectura: yolov8m-seg')
print(f'  Epochs:       {EPOCHS}')
print(f'  Batch:        {BATCH}')
print(f'  ImgSize:      {IMG_SIZE}')
print(f'  Dataset:      {COMBINED}')
print()

results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    patience=PATIENCE,
    device='cuda',
    project=OUT_DIR,
    name='train',
    exist_ok=True,
    val=True,
    save=True,
    plots=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
)

print('\n✅ Entrenamiento completado')

---
## 5. Curvas de entrenamiento y métricas finales

In [ ]:
import pandas as pd

results_dir = Path(OUT_DIR) / 'train'
results_csv = results_dir / 'results.csv'

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    for col in [c for c in df.columns if 'loss' in c.lower() and 'train' in c.lower()]:
        ax.plot(df['epoch'], df[col], label=col.strip())
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.set_title('Train Loss — Modelo D'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

    ax = axes[1]
    for col in [c for c in df.columns if 'map' in c.lower()]:
        ax.plot(df['epoch'], df[col], label=col.strip())
    ax.set_xlabel('Epoch'); ax.set_ylabel('mAP')
    ax.set_title('Validation mAP — Modelo D'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

    plt.suptitle('Curvas de entrenamiento — Modelo D (yolov8m-seg + negativos equilibrados)', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/curvas_entrenamiento_D.png', dpi=120, bbox_inches='tight')
    plt.show()

    print('\n=== MÉTRICAS FINALES MODELO D ===')
    last = df.iloc[-1]
    for col in df.columns:
        if any(k in col.lower() for k in ['map', 'precision', 'recall']):
            print(f'  {col.strip():<40}: {last[col]:.4f}')

    # Comparativa con Modelo C
    print()
    print('=== COMPARATIVA VALIDACIÓN (Roboflow val set) ===')
    print(f'  {"Modelo":<10} {"mAP50(B)":>10} {"mAP50(M)":>10}')
    print(f'  {"C":<10} {0.9556:>10.4f} {0.9526:>10.4f}')
    try:
        map50b = float(df['metrics/mAP50(B)'].iloc[-1])
        map50m = float(df['metrics/mAP50(M)'].iloc[-1])
        print(f'  {"D":<10} {map50b:>10.4f} {map50m:>10.4f}')
    except:
        pass

---
## 6. Guardar pesos en Drive

In [ ]:
best_weights = Path(OUT_DIR) / 'train' / 'weights' / 'best.pt'
dst_weights  = Path(OUT_DIR) / 'weights' / 'best.pt'
dst_weights.parent.mkdir(parents=True, exist_ok=True)

if best_weights.exists():
    shutil.copy2(best_weights, dst_weights)
    size_mb = dst_weights.stat().st_size / 1024 / 1024
    print(f'✅ Pesos guardados: {dst_weights}')
    print(f'   Tamaño: {size_mb:.1f} MB')
else:
    print('❌ No se encontraron pesos')

print()
print('Próximo paso: Notebook 28 — Evaluación Modelo D sobre GAR')
print('Comparativa final: Modelo B vs Modelo C vs Modelo D')

---
## Notas para Notebook 28

**Comparativa final esperada:**

| Modelo | Arquitectura | Negativos train | F1 GAR | FP | FN |
|--------|-------------|----------------|--------|-----|-----|
| B | yolov8m det. | ~3.000 COCO | 0.7949 | 48 | 16 |
| C | yolov8m-seg | 108 LVIS | 0.7963 | 55 | 11 |
| D | yolov8m-seg | 1.586 LVIS+OI | ? | ? | ? |

**Hipótesis específicas para Modelo D:**
- N5 Phone relaxed: debería bajar de 70% (C) hacia los niveles de B (20%)
- N4 Sneaking: debería bajar de 57% (C) — personas sin objeto en OI V7
- N3 Running: debería volver a 0% — personas corriendo en OI V7
- N10 Bottle: debería bajar de 33% (C) — botellas en OI V7